In [ ]:
import pandas as pd
import os

# 父目录路径
base_path = r"\\wsl.localhost\Ubuntu-18.04\home\haibo\pc_side\collected_data\test"

# 设置pandas选项以显示完整数据
pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', None)     # 显示所有行

# 获取所有 seed 文件夹
seed_folders = [f.path for f in os.scandir(base_path) if f.is_dir()]

for seed_folder in seed_folders:
    # 获取 seed 文件夹中的时间戳文件夹
    timestamp_folders = [f.path for f in os.scandir(seed_folder) if f.is_dir()]

    for timestamp_folder in timestamp_folders:
        # 定义文件路径
        file_path = timestamp_folder + "/"

        # 检查所需的文件是否存在
        if not (os.path.exists(file_path + 'trajectory_monitoring_position.csv') and
                os.path.exists(file_path + 'trajectory_monitoring_temperature.csv') and
                os.path.exists(file_path + 'trajectory_monitoring_voltage.csv')):
            print(f"Skipping folder {timestamp_folder}: required files not found.")
            continue

        # 读取三个CSV文件
        position_df = pd.read_csv(file_path + 'trajectory_monitoring_position.csv')
        temperature_df = pd.read_csv(file_path + 'trajectory_monitoring_temperature.csv')
        voltage_df = pd.read_csv(file_path + 'trajectory_monitoring_voltage.csv')

        # 合并数据框，假设它们都有相同的时间列
        merged_df = position_df.merge(temperature_df, on='time')
        merged_df = merged_df.merge(voltage_df, on='time')

        # 提取6个电机的数据并分别保存到6个文件中
        for motor in range(1, 7):
            # 处理同名列时，pandas会生成 _x, _y 来区分，因此需要指定具体列
            motor_df = merged_df[['time', 'motor_{}_x'.format(motor), 'motor_{}_y'.format(motor), 'motor_{}'.format(motor)]].copy()
            motor_df.columns = ['time', 'position', 'temperature', 'voltage']
            motor_df.loc[:, 'label'] = 0
            motor_df.to_csv(file_path + 'data_motor_{}.csv'.format(motor), index=False)

        print(f"Files have been created successfully in {timestamp_folder}.")

print("All folders have been processed successfully.")


In [ ]:
import os
import pandas as pd

# 指定文件夹路径
folder_path = r"F:\OneDrive - CentraleSupelec\Courses\data_experiment_collect\dataset_collect_N5\monitor_log"  # 修改为你的文件夹路径

# 获取所有 CSV 文件，并按文件名排序
files = sorted([f for f in os.listdir(folder_path) if f.startswith("monitor_log_") and f.endswith(".csv")])
print(files)

# 读取并合并所有文件
dfs = [pd.read_csv(os.path.join(folder_path, file)) for file in files]
merged_df = pd.concat(dfs, ignore_index=True)

# 保存合并后的文件
merged_df.to_csv(os.path.join(folder_path, "merged_logs.csv"), index=False)

print("合并完成，文件已保存为 merged_logs.csv")


# 数字顺序


In [ ]:
import os
import pandas as pd
import re

# 指定文件夹路径
folder_path = r"F:\OneDrive - CentraleSupelec\PhD\Paper-Fault_detection\codes\data\monitoring_logs_1000"

# 获取所有 monitor_log_*.csv 文件
files = [f for f in os.listdir(folder_path) if f.startswith("monitor_log_") and f.endswith(".csv")]

# 使用正则表达式提取数字部分，并按数字大小排序
files.sort(key=lambda x: int(re.search(r'monitor_log_(\d+)\.csv', x).group(1)))

# 打印排序后的文件名（可选，便于确认顺序）
print(f"共找到 {len(files)} 个文件，按顺序合并中：")
for f in files:
    print(f)

# 读取并合并所有文件
dfs = [pd.read_csv(os.path.join(folder_path, file)) for file in files]
merged_df = pd.concat(dfs, ignore_index=True)

# 保存合并后的文件
output_path = os.path.join(folder_path, "merged_monitor_log.csv")
merged_df.to_csv(output_path, index=False)

print(f"\n✅ 合并完成，文件已保存为：{output_path}")


In [ ]:
import os
import pandas as pd
import re

folder_path = r"F:\OneDrive - CentraleSupelec\Courses\data_experiment_collect\dataset_collect_N6\monitoring_logs_Mar13"
files = [f for f in os.listdir(folder_path) if f.startswith("monitor_log_") and f.endswith(".csv")]
files.sort(key=lambda x: int(re.search(r'monitor_log_(\d+)\.csv', x).group(1)))

bad_files = []

dfs = []
for file in files:
    path = os.path.join(folder_path, file)
    try:
        df = pd.read_csv(path)  # 默认尝试 utf-8
        dfs.append(df)
    except UnicodeDecodeError:
        print(f"❌ 编码错误: {file}")
        bad_files.append(file)

# 合并成功的
if dfs:
    merged_df = pd.concat(dfs, ignore_index=True)
    merged_df.to_csv(os.path.join(folder_path, "merged_monitor_log.csv"), index=False)
    print("✅ 合并完成，文件已保存为 merged_monitor_log.csv")

# 输出有问题的文件
if bad_files:
    print("\n⚠️ 以下文件编码错误无法解析：")
    for f in bad_files:
        print(f)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 读取 CSV 文件
data = pd.read_csv(r'F:\OneDrive - CentraleSupelec\PhD\Paper-Fault_detection\codes\data\target_all_data_b1_cleaned.csv')

# 先划分训练集和剩余（验证+测试）
train_data, remaining_data = train_test_split(data, test_size=0.6, shuffle=False)

# 再将剩下的部分一分为二（验证 + 测试）
val_data, test_data = train_test_split(remaining_data, test_size=0.75, shuffle=False)

# 打印一下各部分大小确认
print(f"Train: {train_data.shape}")
print(f"Validation: {val_data.shape}")
print(f"Test: {test_data.shape}")

# 保存成新的 CSV 文件，包含列名
# train_data.to_csv("target_train_data.csv", index=False)
# val_data.to_csv("source_val_data.csv", index=False)
test_data.to_csv("source_test_data.csv", index=False)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 读取 CSV 文件
data = pd.read_csv(r'F:\OneDrive - CentraleSupelec\PhD\Paper-Fault_detection\codes\data\target_all_data_b1_cleaned.csv')

# 先划分训练集和剩余（验证+测试）
train_data, test_data = train_test_split(data, test_size=0.3, shuffle=False)


# 打印一下各部分大小确认
print(f"Train: {train_data.shape}")
print(f"Test: {test_data.shape}")

# 保存成新的 CSV 文件，包含列名
# train_data.to_csv("target_train_data.csv", index=False)
# val_data.to_csv("source_val_data.csv", index=False)
test_data.to_csv("target_test_data.csv", index=False)